# Tratamento da SONDA (MD e SD) → séries prontas

**Para que serve:** transformar uma variável do MD (10 min) ou do SD (radiação, 1 min) nas **séries prontas** que o motor wavelet lê.
Aqui ficam as decisões sobre o dado: **passo de tempo** (10 min ou média horária) e **falhas**.

- **Entrada:** MD: `02_dados/sonda_unificada/MD/sonda_MD_AAAA.csv` (gerados por `executar/juntar_sonda_por_ano.py`); SD: os `.dat` brutos da pasta "Dados SONDA" (o SD ainda não foi juntado por ano).
- **Saída:** `02_dados/series_prontas/sonda_<variavel>_<passo>.csv` (colunas `tempo`, `valor`; UTC; sem falhas).
- **Depois:** abra `notebooks/wavelet/rodar_wavelet.ipynb` e escolha a série em `SERIE`.

Regras atuais (iguais às do fluxo antigo):
- média por passo: o horário marca o **fim** do intervalo (01:00 = média de 00:10 a 01:00 no MD, de 00:01 a 01:00 no SD);
- falhas de até `INTERPOLAR_ATE` são interpoladas linearmente; falhas maiores **param** o notebook e são listadas.
- **umidade (`humid`)**: antes disso, as falhas que começam logo depois de uma leitura ≥ 99 % são preenchidas com **100 %** (suposição provisória de ar saturado; ver `02_dados/series_prontas/LEIAME_umidade_falhas_saturacao.txt`);
- **SD**: começa em 01/12/2023 (troca do datalogger) ou no primeiro valor válido; `glo` só a partir de 08/02/2024; `lw_calc` vira falha quando a temperatura do corpo do pirgeômetro (`tp_lw_case`) sai do limite físico (ver `sonda.carregar_variavel_sd`).

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "config").is_dir())
sys.path.insert(0, str(RAIZ))

from src.dados import comum, sonda

# ----------------------------- CONFIGURAÇÃO -----------------------------
TIPO = "MD"                    # MD (meteorologia, 10 min) ou SD (radiação, 1 min)
VARIAVEL = "tp_sfc"            # MD: tp_sfc, humid, press... | SD: glo (onda curta), lw_calc (onda longa)
PASSOS = ["10min", "1h"]       # uma série pronta para cada passo
INTERPOLAR_ATE = "60min"       # falhas até esse tamanho são interpoladas
INICIO = None                  # ex.: "2024-01-01" para usar só um trecho
FIM = None
# ------------------------------------------------------------------------

print(sonda.DESCRICOES.get(VARIAVEL, VARIAVEL))

## 1. Leitura (passo original: 10 min no MD, 1 min no SD)

In [ ]:
PASSO_ORIGINAL = sonda.INTERVALO_REGISTRO[TIPO]
if TIPO == "MD":
    serie_bruta = sonda.carregar_variavel_md(VARIAVEL, inicio=INICIO, fim=FIM)
else:
    serie_bruta = sonda.carregar_variavel_sd(VARIAVEL, inicio=INICIO or sonda.INICIO_CR1000X, fim=FIM)
print(f"{len(serie_bruta)} horários de {serie_bruta.index[0]} a {serie_bruta.index[-1]} | {int(serie_bruta.isna().sum())} sem dado")
comum.falhas(serie_bruta, PASSO_ORIGINAL)

### 1b. Umidade: falhas com o sensor saturado (só para `humid`)

Entre 25/03 e 09/05/2025 o sensor de umidade grava `NAN` por até 15 h, quase sempre à noite, logo depois de ler 100 %.
**Decisão provisória (11/2026):** supor ar saturado e preencher com 100 % toda falha cujo último registro antes dela seja ≥ `LIMIAR_SATURACAO`.
As demais falhas seguem a regra normal. Motivos e ressalvas: `02_dados/series_prontas/LEIAME_umidade_falhas_saturacao.txt`.

In [ ]:
LIMIAR_SATURACAO = 99.0   # % no registro anterior à falha

if VARIAVEL == "humid":
    serie_bruta, saturadas = sonda.preencher_umidade_saturada(serie_bruta, limiar=LIMIAR_SATURACAO)
    print(f"{len(saturadas)} falhas ({saturadas['registros'].sum()} registros) preenchidas com 100 % | "
          f"{int(serie_bruta.isna().sum())} registros ainda sem dado")
    display(saturadas)

## 2. Passo, falhas e gravação

Para cada passo: muda o passo (se for maior que 10 min), mostra as falhas, interpola as curtas e grava a série pronta.

In [ ]:
series_prontas = {}
for passo in PASSOS:
    if pd.Timedelta(passo) == PASSO_ORIGINAL:
        serie, agregacao = serie_bruta, None
    else:
        serie, agregacao = comum.media_por_passo(serie_bruta, passo, PASSO_ORIGINAL)
    serie.name = f"sonda_{VARIAVEL}_{passo}"
    tabela = comum.falhas(serie, passo)
    if len(tabela):
        display(tabela)
    pronta, preenchimento = comum.preencher_falhas(serie, passo, INTERPOLAR_ATE)
    caminho = comum.salvar_serie_pronta(pronta, serie.name)
    series_prontas[serie.name] = pronta

    texto = (f"{serie.name}: {len(pronta)} pontos | {preenchimento['registros_interpolados']} registros "
             f"interpolados (maior falha: {preenchimento['maior_falha']})")
    if agregacao:
        texto += f" | {agregacao['blocos_incompletos']} médias com menos de {agregacao['registros_por_bloco']} registros"
    print(texto)
    print("  ->", caminho)

## 3. Conferência visual das séries prontas

In [ ]:
fig, eixos = plt.subplots(len(series_prontas), 1, figsize=(15, 3.2 * len(series_prontas)), sharex=True, squeeze=False)
for eixo, (nome, pronta) in zip(eixos[:, 0], series_prontas.items()):
    eixo.plot(pronta.index, pronta.values, linewidth=0.4)
    eixo.set_title(nome, fontsize=10)
    eixo.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()